## Step 1: Install and Import Libraries

We only need OpenAI for embeddings and NumPy for calculations.

In [1]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Create Sample Q&A Cache

Let's create a simple cache with 5 question-answer pairs.

In [2]:
# HelpParcel Cached FAQ Dataset

cache = [
    {
        "question": "Where is my order?",
        "answer": "You can track your order by entering your tracking number on the Track Order page. Tracking updates may take up to 24-48 hours after shipment."
    },
    {
        "question": "How long does delivery take?",
        "answer": "Standard shipping typically takes 3-7 business days, while express shipping takes 1-3 business days depending on your location."
    },
    {
        "question": "Can I cancel my order?",
        "answer": "Orders can only be canceled before they are shipped. Once shipped, cancellation is no longer possible, but you may request a return after delivery."
    },
    {
        "question": "How do I request a refund?",
        "answer": "To request a refund, visit your order details page and select 'Request Refund'. Refunds are processed after the returned item passes inspection."
    },
    {
        "question": "My item arrived damaged. What should I do?",
        "answer": "Please submit photos of the damaged item within 48 hours of delivery. Our support team will review the case and arrange a replacement or refund."
    },
    {
        "question": "Can I change my shipping address?",
        "answer": "Shipping addresses can only be modified before the order is shipped. Visit your order page and select 'Edit Address' if available."
    },
    {
        "question": "Why is my tracking number not updating?",
        "answer": "Tracking updates may take 24-48 hours after shipment. If there are no updates after 5 business days, please contact support."
    },
    {
        "question": "Do you offer replacements?",
        "answer": "Yes, replacements are available for damaged, defective, or incorrect items reported within 7 days of delivery."
    },
    {
        "question": "Where can I find my order ID?",
        "answer": "Your order ID is available in your order confirmation email and in the 'My Orders' section of your account."
    },
    {
        "question": "What courier do you use?",
        "answer": "We work with multiple shipping partners depending on the destination. The assigned courier will appear in your order tracking details."
    }
]

## Step 3: Create Embeddings for Cached Questions

Convert each question into a vector (embedding) using OpenAI.

In [4]:
models = ["text-embedding-ada-002", "text-embedding-3-small", "text-embedding-3-large"]

def get_embedding(text):
    """Get embedding for a text using OpenAI."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)

# Create embeddings for all cached questions
print("🔢 Creating embeddings for cached questions...")
for pair in cache:
    pair['embedding'] = get_embedding(pair['question'])
    
# print(f"✅ Created {len(cache)} embeddings")
# print(f"   Embedding dimension: {len(cache[0]['embedding'])}")
# print(f"   First few values: {cache[0]['embedding'][:5]}")
print(f"   Updated cache list: {cache}")

🔢 Creating embeddings for cached questions...
   Updated cache list: [{'question': 'Where is my order?', 'answer': 'You can track your order by entering your tracking number on the Track Order page. Tracking updates may take up to 24-48 hours after shipment.', 'embedding': array([ 0.01586914, -0.02125549, -0.03001404, ..., -0.03594971,
        0.00893402, -0.00481033], shape=(1536,))}, {'question': 'How long does delivery take?', 'answer': 'Standard shipping typically takes 3-7 business days, while express shipping takes 1-3 business days depending on your location.', 'embedding': array([-0.01300812, -0.00444031,  0.06787109, ...,  0.02468872,
        0.0033493 ,  0.02235413], shape=(1536,))}, {'question': 'Can I cancel my order?', 'answer': 'Orders can only be canceled before they are shipped. Once shipped, cancellation is no longer possible, but you may request a return after delivery.', 'embedding': array([ 0.03317261,  0.02120972, -0.04837036, ..., -0.00545883,
       -0.00575256, 

## Step 4: Define Similarity Functions

We'll use cosine distance to measure how similar two questions are.

In [5]:
def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

def cosine_distance(a, b):
    """Calculate cosine distance (1 - similarity)."""
    return 1 - cosine_similarity(a, b)

# Test with two identical questions
test_embedding = cache[0]['embedding']
distance = cosine_distance(test_embedding, test_embedding)
print(f"✅ Functions defined!")
print(f"   Distance between identical questions: {distance:.6f}")
print(f"   (Should be ~0.0)")

✅ Functions defined!
   Distance between identical questions: -0.000000
   (Should be ~0.0)


## Step 5: Implement Cache Lookup Function

This function searches the cache for similar questions.

In [14]:
def search_cache(query, threshold=0.3):
    """
    Search cache for similar questions.
    
    Args:
        query: User's question
        threshold: Maximum distance for a match (lower = stricter)
    
    Returns:
        dict with match info or None
    """
    # Get embedding for the query
    query_embedding = get_embedding(query)
    
    # Calculate distances to all cached questions
    results = []
    for pair in cache:
        distance = cosine_distance(query_embedding, pair['embedding'])
        similarity = 1 - distance
        results.append({
            'question': pair['question'],
            'answer': pair['answer'],
            'distance': distance,
            'similarity': similarity
        })
    
    # Sort by distance (closest first)
    results.sort(key=lambda x: x['distance'])
    
    # Return best match if within threshold
    best_match = results[0]
    if best_match['distance'] <= threshold:
        return {
            'hit': True,
            'matched_question': best_match['question'],
            'answer': best_match['answer'],
            'distance': best_match['distance'],
            'similarity': best_match['similarity'],
            'all_distances': [(r['question'], r['distance']) for r in results]
        }
    else:
        return {
            'hit': False,
            'best_match': best_match['question'],
            'distance': best_match['distance'],
            'similarity': best_match['similarity'],
            'all_distances': [(r['question'], r['distance']) for r in results]
        }

print("✅ Cache search function ready!")

✅ Cache search function ready!


## Step 6: Test Cache Hits (Exact and Semantic Matches)

Let's try some queries and see cache hits!

In [ ]:
# Test:
print("="*60)
print("="*60)

query1 = "Can you check my order status?"
# query1 = "Where is my order?" 100% match
# query1 = "What is python?" CACHE MISS
result1 = search_cache(query1, threshold=0.2)

print(f"\n🔍 Query: {query1}")
if result1['hit']:
    print(f"✅ CACHE HIT!")
    print(f"   Matched: {result1['matched_question']}")
    print(f"   Distance: {result1['distance']:.4f}")
    print(f"   Similarity: {result1['similarity']:.2%}")
    print(f"\n💬 Answer: {result1['answer']}")
else:
    print(f"❌ CACHE MISS")


🔍 Query: Can you check my order status?
❌ CACHE MISS


## Experiment with Different Thresholds

See how threshold affects matching!

In [ ]:
print("="*60)
print("THRESHOLD EXPERIMENTS")
print("="*60)

test_query = "Can you check my order status?"
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

print(f"\n🔍 Query: {test_query}\n")

for threshold in thresholds:
    result = search_cache(test_query, threshold=threshold)
    status = "✅ HIT" if result['hit'] else "❌ MISS"
    print(f"Threshold {threshold:.1f}: {status} (distance: {result['distance']:.4f})")

THRESHOLD EXPERIMENTS

🔍 Query: Can you check my order status?

Threshold 0.1: ❌ MISS (distance: 0.2866)
Threshold 0.2: ❌ MISS (distance: 0.2865)
Threshold 0.3: ✅ HIT (distance: 0.2865)
Threshold 0.4: ✅ HIT (distance: 0.2865)
Threshold 0.5: ✅ HIT (distance: 0.2865)
Threshold 0.6: ✅ HIT (distance: 0.2865)
